In [1]:
# PDF/DOCX: uv add pypdf python-docx
import os
import uuid

import gradio as gr
import numpy as np
import tiktoken
from docx import Document as DocxDocument
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from pypdf import PdfReader


In [2]:
load_dotenv()

MODEL = "gpt-4.1-nano"
EMBED_MODEL = "text-embedding-3-small"
COLLECTION_NAME = "user_upload_rag"
MAX_FILE_MB = 20
CHUNK_SIZE = 200
CHUNK_OVERLAP = 40
DEFAULT_TOP_K = 4
DEFAULT_SEARCH_TYPE = "mmr"

api_key = os.getenv("OPENAI_API_KEY")


In [3]:
def parse_pdf(path: str) -> str:
    reader = PdfReader(path)
    parts = []
    for page in reader.pages:
        parts.append(page.extract_text() or "")
    return "\n\n".join(parts).strip()


def parse_docx(path: str) -> str:
    doc = DocxDocument(path)
    parts = [p.text for p in doc.paragraphs if p.text.strip()]
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                if cell.text.strip():
                    parts.append(cell.text)
    return "\n\n".join(parts).strip()


def load_document_text(file_path: str) -> str:
    path = file_path if isinstance(file_path, str) else (file_path.name if hasattr(file_path, "name") else str(file_path))
    path = path.strip()
    lower = path.lower()
    if lower.endswith(".pdf"):
        return parse_pdf(path)
    if lower.endswith(".docx"):
        return parse_docx(path)
    if lower.endswith(".doc"):
        return parse_docx(path)  # try docx reader; may fail for legacy .doc
    raise ValueError(f"Unsupported format. Use .pdf or .docx (got {path})")


In [4]:
if api_key:
    embeddings = OpenAIEmbeddings(model=EMBED_MODEL, api_key=api_key)
    print("Using OpenAI embeddings")
else:
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    print("Using HuggingFace embeddings fallback")


Using OpenAI embeddings


In [5]:
enc = tiktoken.get_encoding("cl100k_base")


def chunk_text_token_based(text: str, chunk_size: int, overlap: int):
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    tokens = enc.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        end = start + chunk_size
        chunk_tokens = tokens[start:end]
        chunks.append(enc.decode(chunk_tokens))
        if end >= len(tokens):
            break
        start = end - overlap
    return chunks


In [6]:
def ingest_file(file_path, max_mb=MAX_FILE_MB):
    if file_path is None:
        return None, "No file provided."
    path = file_path if isinstance(file_path, str) else getattr(file_path, "name", str(file_path))
    path = path.strip()
    size_mb = os.path.getsize(path) / (1024 * 1024)
    if size_mb > max_mb:
        return None, f"File too large ({size_mb:.1f} MB). Max {max_mb} MB."
    try:
        text = load_document_text(path)
    except Exception as e:
        return None, f"Parse error: {e}"
    if not text.strip():
        return None, "No text extracted from the document."
    name = os.path.basename(path)
    doc_id = f"upload_{uuid.uuid4().hex[:8]}"
    parts = chunk_text_token_based(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
    chunks = [
        {
            "id": f"{doc_id}_chunk_{i}",
            "text": part,
            "metadata": {"source": name, "chunk_index": i, "doc_id": doc_id},
        }
        for i, part in enumerate(parts)
    ]
    lc_docs = [
        Document(page_content=c["text"], metadata={**c["metadata"], "chunk_id": c["id"]})
        for c in chunks
    ]
    collection_name = f"{COLLECTION_NAME}_{uuid.uuid4().hex[:8]}"
    vectorstore = Chroma.from_documents(
        documents=lc_docs,
        embedding=embeddings,
        collection_name=collection_name,
    )
    return vectorstore, f"Ingested {name}: {len(chunks)} chunks."


In [7]:
def build_retriever(vectorstore, search_type: str = DEFAULT_SEARCH_TYPE, k: int = DEFAULT_TOP_K):
    k = max(1, int(k))
    if search_type == "mmr":
        return vectorstore.as_retriever(
            search_type="mmr",
            search_kwargs={"k": k, "fetch_k": max(k * 3, 8), "lambda_mult": 0.6},
        )
    return vectorstore.as_retriever(search_kwargs={"k": k})


def retrieve_with_scores(
    vectorstore, query: str, search_type: str = DEFAULT_SEARCH_TYPE, k: int = DEFAULT_TOP_K
):
    docs = build_retriever(vectorstore, search_type=search_type, k=k).invoke(query)
    scored_results = vectorstore.similarity_search_with_score(query, k=k)
    score_map = {doc.metadata.get("chunk_id"): float(score) for doc, score in scored_results}
    enriched_docs = []
    seen = set()
    for doc in docs:
        cid = doc.metadata.get("chunk_id")
        if cid in seen:
            continue
        seen.add(cid)
        doc.metadata["similarity_score"] = round(score_map.get(cid, float("nan")), 4)
        enriched_docs.append(doc)
    return enriched_docs


In [8]:
def format_sources(retrieved_docs):
    lines = []
    for i, d in enumerate(retrieved_docs, 1):
        source = d.metadata.get("source", "N/A")
        chunk_idx = d.metadata.get("chunk_index", "N/A")
        score = d.metadata.get("similarity_score")
        score_text = f"{score:.4f}" if isinstance(score, (int, float)) and not np.isnan(score) else "N/A"
        lines.append(f"[{i}] {source} | chunk={chunk_idx} | score={score_text}")
    return "\n".join(lines)


def answer_with_rag(
    vectorstore,
    question: str,
    search_type: str = DEFAULT_SEARCH_TYPE,
    k: int = DEFAULT_TOP_K,
):
    retrieved_docs = retrieve_with_scores(vectorstore, question, search_type=search_type, k=k)
    context_blocks = []
    for i, d in enumerate(retrieved_docs, 1):
        context_blocks.append(
            "\n".join(
                [
                    f"Source [{i}]",
                    f"File: {d.metadata.get('source', 'N/A')}",
                    f"Chunk: {d.metadata.get('chunk_index', 'N/A')}",
                    f"Content: {d.page_content}",
                ]
            )
        )
    context = "\n\n---\n\n".join(context_blocks)

    if not api_key:
        return {
            "answer": "No OPENAI_API_KEY. Retrieval works; generation disabled.",
            "contexts": retrieved_docs,
            "sources": format_sources(retrieved_docs),
        }

    llm = ChatOpenAI(model=MODEL, api_key=api_key, temperature=0)
    prompt = f"""You are a retrieval-augmented QA assistant. Answer only from the provided context.
If the answer is supported, respond in 1-3 sentences and end with: Sources: [1], [2] etc.
If the context is missing or unrelated, say: I don't know based on the retrieved context.

Question:
{question}

Context:
{context}
"""
    response = llm.invoke(prompt)
    return {
        "answer": response.content,
        "contexts": retrieved_docs,
        "sources": format_sources(retrieved_docs),
    }


In [ ]:
def process_upload(file, state_vs):
    if file is None:
        return state_vs, "Upload a PDF or Word (.docx) file (max 20 MB)."
    path = file if isinstance(file, str) else (getattr(file, "name", None) or str(file))
    vs, msg = ingest_file(path, max_mb=MAX_FILE_MB)
    if vs is None:
        return state_vs, msg
    return vs, msg


def rag_chat(question, search_type, top_k, state_vs):
    question = (question or "").strip()
    if not question:
        return "Please enter a question.", "", ""
    if state_vs is None:
        return "Upload a document in the Upload tab first.", "", ""

    result = answer_with_rag(state_vs, question, search_type=search_type, k=top_k)
    answer = result.get("answer", "")
    sources = result.get("sources", "")
    contexts = result.get("contexts", [])
    preview = []
    for i, d in enumerate(contexts, 1):
        src = d.metadata.get("source", "N/A")
        chunk_idx = d.metadata.get("chunk_index", "N/A")
        score = d.metadata.get("similarity_score", "N/A")
        snippet = (d.page_content[:300] or "").replace("\n", " ")
        preview.append(f"[{i}] {src}, chunk={chunk_idx}, score={score}\n{snippet}...")
    return answer, sources, "\n\n".join(preview)


with gr.Blocks(title="Upload & RAG") as demo:
    vs_state = gr.State(value=None)
    gr.Markdown("## Document RAG: upload a PDF or DOCX (max 20 MB), then ask questions.")

    with gr.Tab("Upload"):
        file_in = gr.File(
            label="PDF or Word (.docx)",
            file_types=[".pdf", ".docx"],
            type="filepath",
        )
        ingest_btn = gr.Button("Ingest into Chroma")
        status_out = gr.Textbox(label="Status", interactive=False)
        ingest_btn.click(
            fn=process_upload,
            inputs=[file_in, vs_state],
            outputs=[vs_state, status_out],
        )

    with gr.Tab("Ask"):
        q_in = gr.Textbox(lines=3, placeholder="Ask a question about your document...", label="Question")
        search_type_in = gr.Radio(choices=["mmr", "similarity"], value=DEFAULT_SEARCH_TYPE, label="Retriever")
        top_k_in = gr.Slider(minimum=2, maximum=8, step=1, value=DEFAULT_TOP_K, label="Top-k")
        ask_btn = gr.Button("Ask")
        answer_out = gr.Textbox(label="Answer", interactive=False)
        sources_out = gr.Textbox(label="Sources", interactive=False)
        debug_out = gr.Textbox(label="Retrieved chunks", interactive=False)

    ask_btn.click(
        fn=rag_chat,
        inputs=[q_in, search_type_in, top_k_in, vs_state],
        outputs=[answer_out, sources_out, debug_out],
    )

demo.launch(max_file_size=f"{MAX_FILE_MB}mb")
